# Example 10.4.2 / 2: Check a Jump Written with if-then-else

Companion notebook for **10.4.2 Discontinuous Piecewise-Linear (DPWL) Relations** in the AMPL lab Documentation, section "Check a Jump Written with if-then-else". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Check a Jump Written with if-then-else

A jump can also be written as an `if` expression on variables, which MP solver drivers reformulate
with a binary variable and indicator constraints. The strict comparison in the condition becomes a
nonstrict one with a margin: `2*x + 3*y > 5` is passed as `2*x + 3*y >= 5 + eps`, where `eps` is
the driver option `cvt:mip:eps` (aliases `cvt:cmp:eps` and `cmp:eps`). With the margin set to 0,
the solver's version of the condition includes the boundary point $2x + 3y = 5$, while AMPL
evaluates `5 > 5` as false and uses the `else` branch. At that point the solver and AMPL can
disagree about the branch, and the objective AMPL recomputes can differ from the value the solver
optimized.

The AMPL Colab notebook
[Solution check: discontinuous objective function](https://colab.ampl.com/notebooks/solution-check-discontinuous-objective-function.html)
recommends two safeguards. An explicit objective variable `obj_val`, defined by an equality
constraint, makes AMPL report the solver's value instead of recomputing the discontinuous
expression. The driver option `chk:mode=1023` turns on all solution checks, including the
"idealistic" recomputation of reformulated constraints from scratch; a check that fails prints a
warning with the solve message.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
var x >= 0, <= 1;  # first activity level
var y >= 0, <= 1;  # second activity level
var b binary;  # 1 forces the capacity rule
var obj_val;  # the objective value as the solver sees it

subject to ConImpl: b ==> 2*x + 3*y <= 5;  # if b = 1, stay within capacity
subject to TotalIf_con:  # jump of -25 when 2x + 3y exceeds 5
    obj_val == if 2*x + 3*y > 5 then 2*x + 3*y - 3*b - 25 else 2*x + 3*y - 3*b;
minimize TotalIf_var: obj_val;

Writing model.mod


### amplpy: run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# run.py
ampl.option["highs_options"] = "cmp:eps=0 chk:mode=1023"
ampl.solve(solver="highs")
print(ampl.solve_result)
print(ampl.var["obj_val"].value(), ampl.get_value("2*x + 3*y"), ampl.var["b"].value())

HiGHS 1.14.0:   cvt:mip:eps = 0
  sol:chk:mode = 1023
HiGHS 1.14.0: optimal solution; objective -3
0 simplex iterations
0 branching nodes
solved
-3.0 0 1.0


### Pure AMPL: model.run

The same in pure AMPL:

In [5]:
%%ampl_eval
reset;
reset options;
model model.mod;

In [6]:
%%ampl_eval
# model.run
option solver highs;
option highs_options 'cmp:eps=0 chk:mode=1023';
solve;
display obj_val, 2*x + 3*y, b;

HiGHS 1.14.0:   cvt:mip:eps = 0
  sol:chk:mode = 1023
HiGHS 1.14.0: optimal solution; objective -3
0 simplex iterations
0 branching nodes
obj_val = -3
2*x + 3*y = 0
b = 1

